In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import uuid

np.random.seed(42)
random.seed(42)

# -----------------------------
# CONFIG
# -----------------------------
N_ACCOUNTS = 200
N_TXNS = 8000
MULE_RATIO = 0.15
CHANNELS = ['CASH', 'UPI', 'IMPS', 'NEFT', 'RTGS', 'WALLET']
BANKS = ['HDFC', 'ICICI', 'SBI', 'AXIS', 'PAYTM']

START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 2, 1)

# -----------------------------
# ACCOUNTS
# -----------------------------
accounts = []

for i in range(N_ACCOUNTS):
    is_mule = np.random.rand() < MULE_RATIO
    accounts.append({
        "account_id": f"ACCT_{i:04d}",
        "customer_id": f"CUST_{i:04d}",
        "account_open_date": START_DATE - timedelta(days=np.random.randint(30, 2000)),
        "account_type": random.choice(["SAVINGS", "CURRENT"]),
        "risk_rating": "HIGH" if is_mule else random.choice(["LOW", "MEDIUM"]),
        "occupation": random.choice(["SALARIED", "STUDENT", "SELF_EMP", "UNKNOWN"]),
        "expected_turnover": np.random.randint(50000, 500000),
        "is_mule": is_mule
    })

accounts_df = pd.DataFrame(accounts)

# -----------------------------
# TRANSACTIONS
# -----------------------------
txns = []

def random_date():
    return START_DATE + timedelta(
        seconds=np.random.randint(0, int((END_DATE - START_DATE).total_seconds()))
    )

for acct in accounts_df.itertuples():
    base_txn_count = np.random.randint(10, 40)

    for _ in range(base_txn_count):
        amount = np.random.randint(500, 20000)

        txns.append({
            "txn_id": str(uuid.uuid4()),
            "account_id": acct.account_id,
            "txn_timestamp": random_date(),
            "amount": amount,
            "direction": random.choice(["CREDIT", "DEBIT"]),
            "counterparty_account": f"CP_{np.random.randint(1000,9999)}",
            "counterparty_bank": random.choice(BANKS),
            "channel": random.choice(CHANNELS),
            "txn_type": random.choice(["TRANSFER", "CASH", "P2P"])
        })

    # -----------------------------
    # Inject mule patterns
    # -----------------------------
    if acct.is_mule:

        pattern = random.choice(["A", "B", "C", "D", "E"])

        if pattern == "A":  # Rapid pass-through
            t = random_date()
            amt = np.random.randint(20000, 50000)

            txns += [
                {
                    "txn_id": str(uuid.uuid4()),
                    "account_id": acct.account_id,
                    "txn_timestamp": t,
                    "amount": amt,
                    "direction": "CREDIT",
                    "counterparty_account": "SRC_BIG",
                    "counterparty_bank": "UNKNOWN",
                    "channel": "IMPS",
                    "txn_type": "TRANSFER"
                },
                {
                    "txn_id": str(uuid.uuid4()),
                    "account_id": acct.account_id,
                    "txn_timestamp": t + timedelta(minutes=30),
                    "amount": amt * 0.98,
                    "direction": "DEBIT",
                    "counterparty_account": "DEST_BIG",
                    "counterparty_bank": "UNKNOWN",
                    "channel": "IMPS",
                    "txn_type": "TRANSFER"
                }
            ]

        elif pattern == "B":  # Fan-in
            t = random_date()
            for i in range(5):
                txns.append({
                    "txn_id": str(uuid.uuid4()),
                    "account_id": acct.account_id,
                    "txn_timestamp": t + timedelta(minutes=i * 5),
                    "amount": np.random.randint(3000, 8000),
                    "direction": "CREDIT",
                    "counterparty_account": f"SENDER_{i}",
                    "counterparty_bank": random.choice(BANKS),
                    "channel": "UPI",
                    "txn_type": "P2P"
                })

        elif pattern == "C":  # Fan-out
            t = random_date()
            amt = 60000

            txns.append({
                "txn_id": str(uuid.uuid4()),
                "account_id": acct.account_id,
                "txn_timestamp": t,
                "amount": amt,
                "direction": "CREDIT",
                "counterparty_account": "SALARY_SRC",
                "counterparty_bank": "HDFC",
                "channel": "NEFT",
                "txn_type": "TRANSFER"
            })

            for i in range(6):
                txns.append({
                    "txn_id": str(uuid.uuid4()),
                    "account_id": acct.account_id,
                    "txn_timestamp": t + timedelta(minutes=10 * i),
                    "amount": amt / 6,
                    "direction": "DEBIT",
                    "counterparty_account": f"BEN_{i}",
                    "counterparty_bank": random.choice(BANKS),
                    "channel": "IMPS",
                    "txn_type": "TRANSFER"
                })

        elif pattern == "D":  # Dormant → spike
            t = START_DATE + timedelta(days=25)
            txns.append({
                "txn_id": str(uuid.uuid4()),
                "account_id": acct.account_id,
                "txn_timestamp": t,
                "amount": 90000,
                "direction": "CREDIT",
                "counterparty_account": "UNKNOWN",
                "counterparty_bank": "UNKNOWN",
                "channel": "NEFT",
                "txn_type": "TRANSFER"
            })

        elif pattern == "E":  # Channel hopping
            t = random_date()
            amt = 40000
            for ch in ["CASH", "UPI", "IMPS", "WALLET"]:
                txns.append({
                    "txn_id": str(uuid.uuid4()),
                    "account_id": acct.account_id,
                    "txn_timestamp": t,
                    "amount": amt / 4,
                    "direction": "DEBIT",
                    "counterparty_account": f"CH_{ch}",
                    "counterparty_bank": "UNKNOWN",
                    "channel": ch,
                    "txn_type": "TRANSFER"
                })

txns_df = pd.DataFrame(txns)

# -----------------------------
# DERIVED FEATURES
# -----------------------------
def derive_features(df):
    grp = df.groupby("account_id")

    features = grp.apply(lambda x: pd.Series({
        "total_credit": x.loc[x.direction == "CREDIT", "amount"].sum(),
        "total_debit": x.loc[x.direction == "DEBIT", "amount"].sum(),
        "credit_txn_count": (x.direction == "CREDIT").sum(),
        "debit_txn_count": (x.direction == "DEBIT").sum(),
        "unique_senders": x.loc[x.direction == "CREDIT", "counterparty_account"].nunique(),
        "unique_receivers": x.loc[x.direction == "DEBIT", "counterparty_account"].nunique(),
        "channel_entropy": -(x.channel.value_counts(normalize=True)
                             * np.log(x.channel.value_counts(normalize=True) + 1e-9)).sum(),
        "pass_through_ratio": min(
            x.loc[x.direction == "CREDIT", "amount"].sum(),
            x.loc[x.direction == "DEBIT", "amount"].sum()
        ) / (x.amount.sum() + 1e-9)
    }))

    return features.reset_index()

features_df = derive_features(txns_df)

# -----------------------------
# # SAVE
# # -----------------------------
accounts_df.to_csv(r"E:\VS code stuff\Banks Data\mule_pattern\accounts.csv", index=False)
txns_df.to_csv(r"E:\VS code stuff\Banks Data\mule_pattern\transactions.csv", index=False)
# features_df.to_csv("derived_features.csv", index=False)

print("✅ Demo mule dataset generated")


C:\Users\kriss\AppData\Local\Temp\ipykernel_2220\1232565641.py:188: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features = grp.apply(lambda x: pd.Series({


✅ Demo mule dataset generated


In [2]:
accounts_df

,account_id,customer_id,account_open_date,account_type,risk_rating,occupation,expected_turnover,is_mule
0,ACCT_0000,CUST_0000,2022-07-26,SAVINGS,LOW,SELF_EMP,415838,False
1,ACCT_0001,CUST_0001,2020-03-14,SAVINGS,LOW,STUDENT,257892,False
2,ACCT_0002,CUST_0002,2023-08-24,SAVINGS,LOW,UNKNOWN,480410,False
3,ACCT_0003,CUST_0003,2021-02-05,SAVINGS,LOW,SALARIED,225203,False
4,ACCT_0004,CUST_0004,2024-07-25,SAVINGS,HIGH,STUDENT,379365,True
...,...,...,...,...,...,...,...,...
195,ACCT_0195,CUST_0195,2023-09-05,CURRENT,HIGH,SALARIED,109163,True
196,ACCT_0196,CUST_0196,2021-01-25,CURRENT,LOW,UNKNOWN,475523,False
197,ACCT_0197,CUST_0197,2021-02-11,CURRENT,LOW,UNKNOWN,189407,False
198,ACCT_0198,CUST_0198,2021-11-06,CURRENT,MEDIUM,STUDENT,152795,False


In [3]:
txns_df

,txn_id,account_id,txn_timestamp,amount,direction,counterparty_account,counterparty_bank,channel,txn_type
0,17357fc5-78eb-4abb-97e9-4231d8586fd8,ACCT_0000,2025-01-28 14:54:48,1667.0,DEBIT,CP_2062,AXIS,NEFT,P2P
1,df676cac-bce7-4605-878a-cadd6d7b3e9b,ACCT_0000,2025-01-14 04:12:36,19040.0,DEBIT,CP_6161,SBI,UPI,TRANSFER
2,116974a4-07b2-403f-b149-d133cbc8a27e,ACCT_0000,2025-01-11 07:04:25,11674.0,DEBIT,CP_8330,AXIS,UPI,CASH
3,19172b87-b130-49a0-b30a-4652a7a6258e,ACCT_0000,2025-01-01 20:41:00,5834.0,DEBIT,CP_5330,SBI,CASH,CASH
4,342037d3-a600-4eb5-a54e-d76ccef3dab2,ACCT_0000,2025-01-21 13:23:23,7301.0,DEBIT,CP_8241,ICICI,NEFT,TRANSFER
...,...,...,...,...,...,...,...,...,...
4811,555abf8a-f159-40ca-8336-f49530a71b25,ACCT_0199,2025-01-30 19:34:23,5289.0,CREDIT,CP_6350,ICICI,RTGS,CASH
4812,e197a3da-0562-488d-b3ce-c2e08eb689bc,ACCT_0199,2025-01-02 04:19:27,1033.0,DEBIT,CP_1129,AXIS,NEFT,TRANSFER
4813,a8ffee96-922c-4d5d-a543-44b1f0b1ff68,ACCT_0199,2025-01-28 03:37:51,13063.0,CREDIT,CP_2858,HDFC,RTGS,CASH
4814,fb893f41-d4b9-4e01-a853-03b9259d7133,ACCT_0199,2025-01-02 20:03:41,16235.0,DEBIT,CP_4649,AXIS,CASH,P2P


In [4]:
features_df

,account_id,total_credit,total_debit,credit_txn_count,debit_txn_count,unique_senders,unique_receivers,channel_entropy,pass_through_ratio
0,ACCT_0000,31649.0,111551.0,3.0,10.0,3.0,10.0,1.671595,0.221013
1,ACCT_0001,49085.0,94765.0,7.0,9.0,7.0,9.0,1.684373,0.341223
2,ACCT_0002,75178.0,81066.0,8.0,10.0,8.0,10.0,1.739967,0.481158
3,ACCT_0003,137125.0,95979.0,12.0,11.0,12.0,11.0,1.730238,0.411743
4,ACCT_0004,103448.0,163775.0,4.0,16.0,4.0,16.0,1.205216,0.387122
...,...,...,...,...,...,...,...,...,...
195,ACCT_0195,72204.0,125357.0,10.0,10.0,10.0,10.0,1.765057,0.365477
196,ACCT_0196,90184.0,112143.0,9.0,8.0,9.0,8.0,1.599221,0.445734
197,ACCT_0197,194902.0,115372.0,20.0,14.0,20.0,14.0,1.676783,0.371839
198,ACCT_0198,203191.0,181507.0,18.0,14.0,18.0,14.0,1.724972,0.471817


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
from synthetic_data import SyntheticDataGenerator  # <-- yahi tumhara class file

# =============================
# CONFIG
# =============================
N_TXNS = 8000
MULE_RATIO = 0.15

# =============================
# GENERATE DATA (ONLY THIS ENGINE)
# =============================
gen = SyntheticDataGenerator(seed=42)
tx_df, acc_df = gen.generate_transactions(
    num_records=N_TXNS,
    mule_ratio=MULE_RATIO
)

# =============================
# FORMAT TRANSACTIONS (tumhare format me)
# =============================
tx_df = tx_df.rename(columns={
    "transaction_id": "txn_id",
    "timestamp": "txn_timestamp",
    "transaction_type": "txn_type"
})

tx_df["direction"] = tx_df["direction"].map({
    "inbound": "CREDIT",
    "outbound": "DEBIT"
})

BANKS = ['HDFC', 'ICICI', 'SBI', 'AXIS', 'PAYTM']
tx_df["counterparty_bank"] = np.random.choice(BANKS, size=len(tx_df))

transactions_df = tx_df[[
    "txn_id",
    "account_id",
    "txn_timestamp",
    "amount",
    "direction",
    "counterparty_account",
    "counterparty_bank",
    "channel",
    "txn_type",

    # ===== AML SIGNALS FROM YOUR GENERATOR =====
    "is_suspicious",
    "mule_pattern",
    "hour",
    "day_of_week",
    "is_weekend",
    "is_night",
    "device_id",
    "ip_address",
    "geo_location",
    "balance_after"
]]

# =============================
# FORMAT ACCOUNTS (tumhare format me)
# =============================
acc_df["customer_id"] = acc_df["account_id"].str.replace("ACC", "CUST")
acc_df["risk_rating"] = np.where(acc_df["is_mule"], "HIGH", "LOW")
acc_df["occupation"] = np.random.choice(
    ["SALARIED", "STUDENT", "SELF_EMP", "UNKNOWN"],
    size=len(acc_df)
)
acc_df["expected_turnover"] = acc_df["declared_income"].astype(int)

accounts_df = acc_df[[
    "account_id",
    "customer_id",
    "account_open_date",
    "customer_type",
    "risk_rating",
    "occupation",
    "expected_turnover",
    "is_mule"
]]

# =============================
# SAVE
# =============================
accounts_df.to_csv("accounts.csv", index=False)
transactions_df.to_csv("transactions.csv", index=False)

print("✅ Done. Only SyntheticDataGenerator used.")


✅ Done. Only SyntheticDataGenerator used.


In [2]:
import pandas as pd
tx = pd.read_csv(r'E:\VS CODE Backup\Trae\AI_AML_tool\data_generation_scripts\transactions.csv')
acc = pd.read_csv(r'E:\VS CODE Backup\Trae\AI_AML_tool\data_generation_scripts\accounts.csv')

In [3]:
tx.columns

Index(['txn_id', 'account_id', 'txn_timestamp', 'amount', 'direction',
       'counterparty_account', 'counterparty_bank', 'channel', 'txn_type',
       'is_suspicious', 'mule_pattern', 'hour', 'day_of_week', 'is_weekend',
       'is_night', 'device_id', 'ip_address', 'geo_location', 'balance_after'],
      dtype='object')

In [4]:
acc.columns

Index(['account_id', 'customer_id', 'account_open_date', 'customer_type',
       'risk_rating', 'occupation', 'expected_turnover', 'is_mule'],
      dtype='object')